# 参数管理

In [27]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
# 整体
print(net)

# 某层
print(net[2])

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)
Linear(in_features=8, out_features=1, bias=True)


## 参数访问

### 一次性访问所有参数


In [47]:
# 所有层的(没有ReLU)
print(net.state_dict())
print(type(net.state_dict()))
print('-----------')


# 某一层
print(net[2].state_dict())
print(type(net[2].state_dict()))

OrderedDict([('0.weight', tensor([[-0.0359, -0.1113, -0.0560, -0.3610],
        [-0.0850,  0.0467,  0.0280, -0.0839],
        [ 0.2441, -0.2452,  0.3193,  0.1794],
        [ 0.0131,  0.3792,  0.4807, -0.1022],
        [-0.0882, -0.0954, -0.2788,  0.0179],
        [ 0.1402,  0.4889,  0.0035,  0.3007],
        [-0.4515, -0.0525,  0.2825, -0.1868],
        [-0.1527,  0.2921, -0.1597,  0.3325]])), ('0.bias', tensor([-0.0653,  0.4274,  0.0800,  0.2193,  0.3678, -0.2354, -0.3849,  0.2034])), ('2.weight', tensor([[ 0.0691,  0.3446,  0.3221, -0.0545,  0.1119,  0.1805, -0.1573, -0.0590]])), ('2.bias', tensor([0.0903]))])
<class 'collections.OrderedDict'>
-----------
OrderedDict([('weight', tensor([[ 0.0691,  0.3446,  0.3221, -0.0545,  0.1119,  0.1805, -0.1573, -0.0590]])), ('bias', tensor([0.0903]))])
<class 'collections.OrderedDict'>


类型为`<class 'collections.OrderedDict'>`, 这个全连接层包含两个参数，分别是该层的权重weight和偏置bias。两者都存储为单精度浮点数（float32）。

整体输出时名为`0.weight`,单层输出时名为`weight`

In [49]:
for name, param in net.named_parameters():
    print(name, param, sep='\n')
print('-----------')

for name, param in net[2].named_parameters():
    print(name, param, sep='\n')
print('-----------')

# 这个就是传入的不带名字的参数, torch.optim.SGD(net.parameters(), lr=0.1)
for param in net.parameters():
    print(param, sep='\n')

0.weight
Parameter containing:
tensor([[-0.0359, -0.1113, -0.0560, -0.3610],
        [-0.0850,  0.0467,  0.0280, -0.0839],
        [ 0.2441, -0.2452,  0.3193,  0.1794],
        [ 0.0131,  0.3792,  0.4807, -0.1022],
        [-0.0882, -0.0954, -0.2788,  0.0179],
        [ 0.1402,  0.4889,  0.0035,  0.3007],
        [-0.4515, -0.0525,  0.2825, -0.1868],
        [-0.1527,  0.2921, -0.1597,  0.3325]], requires_grad=True)
0.bias
Parameter containing:
tensor([-0.0653,  0.4274,  0.0800,  0.2193,  0.3678, -0.2354, -0.3849,  0.2034],
       requires_grad=True)
2.weight
Parameter containing:
tensor([[ 0.0691,  0.3446,  0.3221, -0.0545,  0.1119,  0.1805, -0.1573, -0.0590]],
       requires_grad=True)
2.bias
Parameter containing:
tensor([0.0903], requires_grad=True)
-----------
weight
Parameter containing:
tensor([[ 0.0691,  0.3446,  0.3221, -0.0545,  0.1119,  0.1805, -0.1573, -0.0590]],
       requires_grad=True)
bias
Parameter containing:
tensor([0.0903], requires_grad=True)
-----------
Parameter

### 目标参数

每层的每个参数(`.weight`和`.bias`)都表示为参数类的一个实例 `<class 'torch.nn.parameter.Parameter'>`。


包含:

- 值: `.data`
      

- 梯度: `.grad`

  在上面这个网络中，由于我们还没有调用反向传播，所以参数的梯度处于初始状态`None`。

- 额外信息

  `.device`: 存储在CPU上还是GPU上

In [20]:
# 权重
print(type(net[2].weight))
print(net[2].weight)
# 值
print(net[2].weight.data)
# 梯度
print(net[2].weight.grad)
print()

# 偏置
print(type(net[2].bias))
print(net[2].bias)
# 值
print(net[2].bias.data)
# 梯度
print(net[2].bias.grad)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([[ 0.0458,  0.2153, -0.2498,  0.1410,  0.0375, -0.2264,  0.1967,  0.2280]],
       requires_grad=True)
tensor([[ 0.0458,  0.2153, -0.2498,  0.1410,  0.0375, -0.2264,  0.1967,  0.2280]])
None

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.0729], requires_grad=True)
tensor([-0.0729])
None


In [21]:
# 额外信息
print(net[2].weight.device)

cpu


In [35]:
# 另一种访问网络参数的方式
net.state_dict()['2.weight'].data

tensor([[ 0.0691,  0.3446,  0.3221, -0.0545,  0.1119,  0.1805, -0.1573, -0.0590]])

这为我们提供了另一种访问网络参数的方式，如下所示。


In [6]:
net.state_dict()['2.bias'].data

tensor([0.2871])

### 从嵌套块收集参数

In [60]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1(), nn.Linear(4, 4), block1())
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))
net

Sequential(
  (0): Sequential(
    (0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (1): Linear(in_features=4, out_features=4, bias=True)
    (2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)

In [8]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


因为层是分层嵌套的，所以我们也可以像通过嵌套列表索引一样访问它们。
下面，我们访问第一个主要的块中、第二个子块的第一层的偏置项。


In [64]:
net[0][0][0].weight.data

tensor([[ 0.0227, -0.0973, -0.4707,  0.4732],
        [ 0.3803, -0.3615, -0.0491,  0.2717],
        [ 0.2965,  0.3781,  0.0135,  0.1115],
        [ 0.2983,  0.2822,  0.2246, -0.4244],
        [-0.4664, -0.2939,  0.3222, -0.4690],
        [ 0.3131,  0.2995,  0.0045, -0.2128],
        [-0.1545, -0.1798,  0.4078,  0.4646],
        [-0.1917,  0.0998, -0.1750, -0.3558]])

In [65]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1(), nn.Linear(4, 4), block1())
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))

# 检查参数是否相同
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])
net[0][0][0].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])

tensor([False, False, False, False])
tensor([False, False, False, False])


## 参数初始化

### 标准初始化

> 方法1

In [ ]:
def init_weights(m):
    if type(m) == nn.Linear:
        # 以均值0和标准差0.01的正态分布, 初始化权重
        nn.init.normal_(m.weight, mean=0, std=0.01)
        # 全0初始化偏置
        nn.init.zeros_(m.bias)


# 应用到网络上
net.apply(init_weights)

# 学习率为0.1的小批量随机梯度下降作为优化算法
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)

> 标准总结

In [ ]:
# 以均值0和标准差0.01的正态分布
nn.init.normal_(m.weight, mean=0, std=0.01)
# 均居分布
nn.init.uniform_(m.weight, -10, 10)
# 全0初始化
nn.init.zeros_(m.weight)
# 给定的常数
nn.init.constant_(m.weight, 1)

### 自定义初始化

> 方法1

在下面的例子中，我们使用以下的分布为任意权重参数$w$定义初始化方法：

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \text{ 可能性 } \frac{1}{4} \\
            0    & \text{ 可能性 } \frac{1}{2} \\
        U(-10, -5) & \text{ 可能性 } \frac{1}{4}
    \end{cases}
\end{aligned}
$$


同样，我们实现了一个`my_init`函数来应用到`net`。


In [71]:
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5


net.apply(init_weights)

Sequential(
  (0): Sequential(
    (0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (1): Linear(in_features=4, out_features=4, bias=True)
    (2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)

> 方法2: 直接设置参数。


In [74]:
net[1].weight.data[:] += 1
net[1].weight.data[0, 0] = 42
net[1].weight.data[0]

tensor([42.0000,  1.0000, -4.2669,  1.0000])

> 方法3: 

In [73]:
num_inputs, num_outputs, num_hiddens = 784, 10, 256

W1 = nn.Parameter(torch.randn(
    num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(
    num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

optimizer = torch.optim.SGD(params, lr=0.01)

## 参数绑定

有时我们希望在多个层间共享参数：
我们可以定义一个稠密层，然后使用它的参数来设置另一个层的参数。


In [58]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


这个例子表明第三个和第五个神经网络层的参数是绑定的。
它们不仅值相等，而且由相同的张量表示。
因此，如果我们改变其中一个参数，另一个参数也会改变。
你可能会思考：当参数绑定时，梯度会发生什么情况？
答案是由于模型参数包含梯度，因此在反向传播期间第二个隐藏层
（即第三个神经网络层）和第三个隐藏层（即第五个神经网络层）的梯度会加在一起。


> 嵌套块的易錯区分

In [65]:
# 这不是共享的
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1(), nn.Linear(4, 4), block1())
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))

# 检查参数是否相同
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])
net[0][0][0].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])

tensor([False, False, False, False])
tensor([False, False, False, False])


In [67]:
# 这是共享的
block1 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                       nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1, nn.Linear(4, 4), block1)
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))

# 检查参数是否相同
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])
net[0][0][0].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])

tensor([True, True, True, True])
tensor([True, True, True, True])
